# Constructive CLM-005 — Kaggle Formal Run + Publisher

This notebook runs the **frozen formal decision** for CLM-005 and then publishes the exact canonical artifacts back to the registered GitHub branch.

- Development/CI never runs formal seeds.
- Formal seeds: `90811 / 90812 / 90813`.
- Kaggle Secret required: `GITHUB_TOKEN` with push access to `ArcheLabs/mini-cells`.
- The publisher commits **only** `artifacts/experiments/constructive-clm-005-scaffold-removal/`.
- If a canonical CLM-005 decision is already tracked, publication refuses to run again.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess

REPO = Path("/kaggle/working/mini-cells-clm005")
BRANCH = "codex/constructive-clm-005-endogenous-control"
REMOTE = "https://github.com/ArcheLabs/mini-cells.git"

if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(
    ["git", "clone", "--branch", BRANCH, REMOTE, str(REPO)],
    check=True,
)

os.chdir(REPO)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)

head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
protocol = REPO / "research/validations/constructive-clm-005-scaffold-removal/protocol.json"
protocol_sha = hashlib.sha256(protocol.read_bytes()).hexdigest()

tracked_decision = subprocess.run(
    [
        "git", "ls-files", "--error-unmatch",
        "artifacts/experiments/constructive-clm-005-scaffold-removal/decision.json",
    ],
    text=True,
    capture_output=True,
)

assert tracked_decision.returncode != 0, (
    "Canonical CLM-005 formal artifacts are already tracked on this branch. "
    "Refusing to consume the frozen formal seeds again."
)

print("Branch:", BRANCH)
print("Commit:", head)
print("Protocol SHA-256:", protocol_sha)
print("Frozen formal seeds: 90811 / 90812 / 90813")


In [ ]:
# Run the registered formal decision exactly once.
subprocess.run(
    [
        "python",
        "scripts/research/run_constructive_clm_005.py",
        "--formal",
    ],
    check=True,
)

decision_path = (
    REPO
    / "artifacts/experiments/constructive-clm-005-scaffold-removal/decision.json"
)
payload = json.loads(decision_path.read_text(encoding="utf-8"))

assert payload["scientific_decision"] is True
assert payload["completed_seeds"] == [90811, 90812, 90813]
assert payload["missing_seeds"] == []
assert payload["protocol_sha256"] == protocol_sha
assert payload["status"] in {
    "LEARNED_CONTROL_PLANE_TRANSITION_SUPPORTED",
    "LEARNED_CONTROL_PLANE_TRANSITION_NOT_SUPPORTED",
}

print("\n=== CLM-005 FORMAL DECISION ===")
print(json.dumps({
    "status": payload["status"],
    "scientific_decision": payload["scientific_decision"],
    "completed_seeds": payload["completed_seeds"],
    "missing_seeds": payload["missing_seeds"],
    "protocol_sha256": payload["protocol_sha256"],
}, indent=2))

print("\n=== PER-SEED GATES ===")
for result in payload["results"]:
    print("\nseed:", result["seed"], "pass:", result["pass"])
    for gate, passed in result["gates"].items():
        print(f"  {gate}: {passed}")


In [ ]:
# Publish the exact first formal artifacts back to GitHub.
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
assert token, "Kaggle Secret GITHUB_TOKEN is missing."

env = os.environ.copy()
env["GITHUB_TOKEN"] = token

subprocess.run(
    [
        "python",
        "scripts/research/publish_constructive_clm_005.py",
        "--branch",
        BRANCH,
        "--token-env",
        "GITHUB_TOKEN",
    ],
    check=True,
    env=env,
)

published_head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()

print("\nCLM-005 formal artifacts published.")
print("branch:", BRANCH)
print("commit:", published_head)
print("status:", payload["status"])
